# Lab: DiD foundations with Kentucky workers’ compensation (Python)

[View this lab on the QED Labs website](https://defenceeconomist.github.io/qedlabs/labs/difference-in-differences-foundations-python-lab.html)

## How to use this lab

Allow 45–60 minutes. Basic regression and Python data-frame familiarity are assumed.
Use the [tested environment setup](https://defenceeconomist.github.io/qedlabs/labs/difference-in-differences-reproducibility.html) before running every cell in order.

[R companion](https://defenceeconomist.github.io/qedlabs/labs/difference-in-differences-foundations-lab.html) · [Application catalogue](https://defenceeconomist.github.io/qedlabs/notes/did/difference-in-differences-applications.html)

The code downloads a checksum-verified upstream data file on first use and caches it locally. This is a teaching reproduction, not a replication of every specification in the original paper.

## Research question and design

Did the 1980 increase in Kentucky's workers' compensation cap change benefit duration for high earners relative to low earners? The outcome is log duration. These are repeated cross-sections of claims, not a panel of claimants [@meyer1995workerscomp; @heiss2026didexample].

The target is the ATT for high earners represented by these samples, under parallel untreated trends, no anticipation, and stable composition. Low earners need not have the same outcome level; their untreated change must be credible.

## 1. Load and audit the sample

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyreadr

# Immutable upstream package data; do not silently accept a changed file.
url = "https://raw.githubusercontent.com/cran/wooldridge/5c16676c8c8e6685985d98661c55088c878902be/data/injury.RData"
expected_sha256 = "ec9d7cb7f50140fc64756856b0ad5a76151d13d6ec65ca2c66118df1eea3748f"
cache = Path(".qedlabs-cache")
cache.mkdir(exist_ok=True)
path = cache / "injury.rda"
if not path.exists():
    payload = urlopen(url, timeout=60).read()
    assert hashlib.sha256(payload).hexdigest() == expected_sha256, "Data checksum mismatch"
    path.write_bytes(payload)
assert hashlib.sha256(path.read_bytes()).hexdigest() == expected_sha256, "Cached data changed"
data = pyreadr.read_r(str(path))["injury"]
print({p: metadata.version(p) for p in ["pandas", "numpy", "pyreadr"]})
print(data.shape)

In [ ]:
import statsmodels.formula.api as smf
ky = data.loc[data.ky == 1].copy()
assert len(ky) == 5626
assert ky[['ldurat', 'afchnge', 'highearn']].notna().all().all()
assert (ky.durat > 0).all()
counts = pd.crosstab(ky.highearn, ky.afchnge)
assert (counts > 0).all().all()
print(counts)

Checkpoint: why would a change in who files a claim matter even if group labels stay fixed?

## 2. Four means and the counterfactual

In [ ]:
means = ky.groupby(['highearn', 'afchnge']).ldurat.mean().unstack()
changes = means[1] - means[0]
did_manual = float(changes.loc[1] - changes.loc[0])
print(means)
print({'low_earner_change': changes.loc[0], 'high_earner_change': changes.loc[1],
       'did_log_points': did_manual})
ax = means.T.rename(columns={0: 'Low earners', 1: 'High earners'}).plot(marker='o')
counterfactual = means.loc[1, 0] + changes.loc[0]
ax.plot([0, 1], [means.loc[1, 0], counterfactual], '--', color='black', label='High-earner untreated counterfactual')
ax.set(xticks=[0, 1], xticklabels=['Before', 'After'], ylabel='Mean log benefit duration', xlabel='')
ax.legend(); plt.show()

The dashed endpoint is an assumption-based prediction. One pre-treatment period cannot establish a pre-treatment trend.

## 3. Recover the contrast with regression

In [ ]:
fit = smf.ols('ldurat ~ highearn * afchnge', data=ky).fit(cov_type='HC1')
term = 'highearn:afchnge'
assert abs(fit.params[term] - did_manual) < 1e-10
print(fit.summary().tables[1])
print({'log_point_effect': did_manual,
       'geometric_mean_ratio_percent': 100 * np.expm1(did_manual)})
assert np.isclose(did_manual, 0.1906012, atol=1e-6)

The contrast is about 0.191 log points. Exponentiation gives approximately 21% for the ratio of geometric-mean changes. It is not automatically a 21% change in arithmetic mean weeks or an average individual percentage effect.

HC1 errors are an illustration of claim-level sampling uncertainty. Thousands of claims do not create thousands of independently assigned policies. The single policy contrast limits policy-level inference.

## 4. Covariates and sample composition

In [ ]:
covariates = ['male', 'married', 'age', 'hosp', 'indust', 'injtype', 'lprewage']
common = ky.dropna(subset=covariates).copy()
base_common = smf.ols('ldurat ~ highearn * afchnge', data=common).fit(cov_type='HC1')
adjusted = smf.ols('ldurat ~ highearn * afchnge + male + married + age + hosp + C(indust) + C(injtype) + lprewage', data=common).fit(cov_type='HC1')
print(pd.DataFrame({'model': ['Full sample', 'Common sample', 'Adjusted common sample'],
                    'n': [fit.nobs, base_common.nobs, adjusted.nobs],
                    'estimate': [m.params[term] for m in [fit, base_common, adjusted]]}))

This adjustment is a specification exercise, not an endorsed causal model: hospitalization and injury classification may reflect post-policy behavior or reporting. Explain the timing of each covariate before interpreting the adjusted contrast.

## Worked answers and reporting exercise

1. **Why not compare after-period levels?** The groups already differ before the policy. The low-earner trend supplies the assumed untreated change.
2. **What does the regression add?** It reproduces the four-cell contrast exactly here; it supplies a framework for inference and extensions, not a new source of identification.
3. **Why use a common sample?** Otherwise a coefficient change mixes adjustment with missing-data selection.
4. **What remains unknown?** Without extra periods, pre-trends cannot be examined. Claim composition, simultaneous policies, and behavioral responses need external evidence.

Write a six-sentence conclusion naming the policy, comparison, outcome scale, ATT, identifying assumption, and largest remaining threat. Use the [source notes](https://defenceeconomist.github.io/qedlabs/notes/did/gertler-difference-in-differences-notes.html) to explain what DiD does and does not remove.